# Dashboard Data Preparation and Aggregation

After generating real-time predictions, the next step is to prepare the data for visualization in the Shiny dashboard.

The dashboard requires two levels of data:

1. **Overview Level (All Cities)**
   - One record per city
   - Maximum predicted demand over forecast period

2. **Drill-down Level (Single City)**
   - Full time-series forecast data
   - Detailed weather and prediction information

This notebook reconstructs the aggregation logic used in the Shiny server to prepare data for visualization.

## Import Required Libraries

We use:

- `dplyr`: data manipulation  
- `readr`: data loading  

In [ ]:
# Load required libraries

# dplyr for grouping and summarisation
library(dplyr)

# readr for reading CSV files
library(readr)

## Load Prediction Dataset

We load the dataset generated from the prediction pipeline.

In [ ]:
# Generate full dataset using pipeline function

city_weather_bike_df <- generate_city_weather_bike_data()

# Inspect dataset
head(city_weather_bike_df)

## Understanding Dataset Structure

The dataset contains:

- CITY_ASCII → City name  
- LAT, LNG → Geographic coordinates  
- TEMPERATURE, HUMIDITY → Weather features  
- BIKE_PREDICTION → Predicted demand  
- BIKE_PREDICTION_LEVEL → Demand category  
- LABEL → Short popup label  
- DETAILED_LABEL → Detailed popup label  
- FORECASTDATETIME → Time dimension  

This dataset supports both overview and drill-down views.

In [ ]:
# Display structure of dataset

str(city_weather_bike_df)

## Create Aggregated Dataset (Overview Level)

To display all cities on the map, we create a dataset with:

- One row per city  
- Maximum predicted demand  
- Corresponding label and prediction level  

This logic directly mirrors the Shiny server implementation.

In [ ]:
# Aggregate dataset for overview map

cities_max_bike <- city_weather_bike_df %>%
  
  # Group by city and coordinates
  group_by(CITY_ASCII, LAT, LNG) %>%
  
  # Summarise to get max demand per city
  summarise(
    
    # Maximum bike demand prediction
    BIKE_PREDICTION = max(BIKE_PREDICTION, na.rm = TRUE),
    
    # Corresponding prediction level
    BIKE_PREDICTION_LEVEL = BIKE_PREDICTION_LEVEL[which.max(BIKE_PREDICTION)],
    
    # Short popup label
    LABEL = LABEL[which.max(BIKE_PREDICTION)],
    
    # Detailed popup label
    DETAILED_LABEL = DETAILED_LABEL[which.max(BIKE_PREDICTION)],
    
    .groups = "drop"
  )

# Inspect aggregated dataset
head(cities_max_bike)

## Mapping to Dashboard Behavior

The Shiny dashboard uses two data flows:

### 1. Overview Map (All Cities)
- Uses `cities_max_bike`
- Displays one marker per city
- Marker size and color depend on demand level

### 2. Drill-Down View (Single City)
- Uses full dataset (`city_weather_bike_df`)
- Displays all forecast time points
- Shows detailed weather + demand information

This conditional logic is driven by the dropdown input:

- `input$city_dropdown`

In [ ]:
# Example: Filter data for a specific city (Drill-down view)

selected_city <- city_weather_bike_df %>%
  filter(CITY_ASCII == "London")

# Inspect filtered data
head(selected_city)

## Marker Encoding Logic

The dashboard encodes demand visually using:

- Radius:
  - small → 6  
  - medium → 10  
  - large → 12  

- Color:
  - small → green  
  - medium → yellow  
  - large → red  

This encoding allows users to quickly identify high-demand cities.

In [ ]:
# Example: Create mapping logic for visualization

cities_max_bike <- cities_max_bike %>%
  mutate(
    marker_size = case_when(
      BIKE_PREDICTION_LEVEL == "small" ~ 6,
      BIKE_PREDICTION_LEVEL == "medium" ~ 10,
      BIKE_PREDICTION_LEVEL == "large" ~ 12
    ),
    
    marker_color = case_when(
      BIKE_PREDICTION_LEVEL == "small" ~ "green",
      BIKE_PREDICTION_LEVEL == "medium" ~ "yellow",
      BIKE_PREDICTION_LEVEL == "large" ~ "red"
    )
  )

# Inspect encoded data
head(cities_max_bike)

## Summary

In this notebook, we:

- Prepared prediction data for dashboard visualization
- Created aggregated city-level dataset
- Defined drill-down filtering logic
- Implemented marker encoding for visual representation

This structure directly supports the interactive behavior of the Shiny dashboard.

---

## Author & Acknowledgment

**Author:**  
<span style="color:blue">Deepan Mehta </span>  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on preparing prediction data for visualization in the Shiny dashboard by implementing aggregation and filtering logic.

The workflow follows <span style="color:blue">IBM Skills Network </span> instructional labs on dashboard data preparation and transformation.

Special acknowledgment is given to:

- <span style="color:blue">Yan Luo </span>  
- <span style="color:blue">Jeff Grossman</span>  

---

## Project Context

This notebook represents the data preparation stage for dashboard visualization:

- Data Collection  
- Data Wrangling (ETL)  
- Exploratory Data Analysis (EDA)  
- Model Development  
- Model Evaluation  
- Model Selection  
- API Integration & Prediction Pipeline  
- **Dashboard Data Preparation**  
- Deployment (R Shiny Dashboard)

---

## Notes

This step ensures that prediction outputs are structured efficiently for interactive visualization and user-driven exploration.

---